In [91]:
import ROOT
import numpy as np
import pandas as pd
from scipy.stats import f as f_dist
import math
 
ROOT.gStyle.SetOptFit(1111)    # aggiunge il label con chi2/ndf e probabilita e parametri se metti 0000 toglie i dati
ROOT.gStyle.SetFitFormat(
    "4.2e"
)  # con 2 cifre decimali


ROOT.gROOT.SetBatch(True)

In [92]:
file_path = "esperienza5.xlsx"

sheets = {
    'Dati': 'Dati',
    'Manuele': 'Manuele',
    'Alessio': 'Alessio',
    'Emanuele': 'Emanuele',
    'Michele': 'Michele'
}

def load_data(sheet_name):
    try:
        df = pd.read_excel(file_path, sheet_name=sheet_name, skiprows=0)
        theta = pd.to_numeric(df.iloc[:, 0], errors='coerce')
        print(theta)
        theta = theta.dropna()
        
        theta_val = theta.values.astype(np.float64)
        
        return theta_val
    except Exception as e:
        print(f"Errore nel caricamento del foglio {sheet_name}: {e}")
        return None, None


def IQR(dist):
    return np.percentile(dist, 75) - np.percentile(dist, 25)

In [ ]:
root_objects = []
def make_hist(nomi, title, filename):
    c = ROOT.TCanvas(f"c_{filename}", title, 800, 600)
    c.SetGrid()
    

    for name in nomi:
        if name in data:
            theta = data[name]

            gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(theta), max(theta), 3)
            gaussian_fit.SetParNames("const", "mu", "sigma")
            gaussian_fit.SetLineColor(ROOT.kRed + 1)
            gaussian_fit.SetLineWidth(3)


            delta_bin = 2*IQR(theta)/ (len(theta))**(1/3)
            n_bin= (np.max(theta) - np.min(theta))/delta_bin
            n_bin = int(np.round(n_bin))

            if name == "Manuele" or name == "Michele":
                hist = ROOT.TH1F("hist", f"{title}; θ_B; conteggi", 8, min(theta), max(theta))
                print(theta)
            else:
                 hist = ROOT.TH1F("hist", f"{title}; θ_B; conteggi", n_bin, min(theta), max(theta))
            for val in theta:
                hist.Fill(val)
                leg = ROOT.TLegend(0.15, 0.70, 0.40, 0.85)
                leg.SetBorderSize(1)
                leg.SetFillStyle(1001)

        gaussian_fit.SetParameters(hist.GetMaximum(), hist.GetMean(), hist.GetRMS())
            
        hist.SetLineColor(ROOT.kAzure + 3)
        hist.SetLineWidth(2)
        hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
        hist.GetXaxis().SetTitle("θ_B")
        hist.GetYaxis().SetTitle("conteggi")
        hist.SetTitle(f"{title}")

        r = hist.Fit(gaussian_fit, "S")

        fitStatus = r.Status()
        print(f"Fit status: {fitStatus}")

        parameters = gaussian_fit.GetParameters()
        errors = gaussian_fit.GetParErrors()

        chi2 = gaussian_fit.GetChisquare()
        '''
        print("Parametri stimati:")
        print(f"const = {parameters[0]} ± {errors[0]}")
        print(f"#mu = {parameters[1]} ± {errors[1]}")
        print(f"#sigma = {parameters[2]} ± {errors[2]}")
        print(f"Chi2 = {chi2}")
        print("\n")
        print("--------------------------------------------------------------")
        print("\n")
        '''
        root_objects.append(hist)

    hist.Draw()

    '''
    leg2 = ROOT.TLegend(0, 0.30, 0.15, 0.39)
    leg2.AddEntry(hist, "Residui", "f")
    leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
    leg2.Draw()
    '''

    c.SaveAs(filename)


In [94]:
data = {}
for key, s_name in sheets.items():
    theta = load_data(s_name)
    data[key] = (theta) 


0      56.62500
1      57.08333
2      56.51667
3      57.61667
4      57.11667
         ...   
100    56.70000
101    56.65000
102    57.40833
103    57.30000
104    56.73333
Name: θ_Bi, Length: 105, dtype: float64
0     56.62500
1     57.08333
2     56.51667
3     57.61667
4     57.11667
5     56.58333
6     57.01667
7     56.70000
8     56.45000
9     56.89167
10    57.29167
11    56.57500
12    56.45000
13    56.53333
14    56.70000
15    56.45000
16    56.89167
17    56.45000
18    56.50833
19    57.20000
20    56.70000
21    57.20000
22    57.10000
23    56.74167
24    56.78333
25    57.07500
26    56.37000
27    56.53333
28    57.55000
29    56.61667
30         NaN
31         NaN
32         NaN
33         NaN
34         NaN
35         NaN
36         NaN
37         NaN
38         NaN
39         NaN
40         NaN
41         NaN
42         NaN
43         NaN
44         NaN
45         NaN
46         NaN
47         NaN
48         NaN
49         NaN
50         NaN
51         NaN
52  

In [95]:
make_hist(['Dati'], 'Istogramma con tutti i dati', 'hist_totalData.png')
make_hist(['Manuele'], 'Istogramma con i dati di Manuele', 'histManuData.png')
make_hist(['Alessio'], 'Istogramma con i dati di Alessio', 'histAleData.png')
make_hist(['Emanuele'], 'Istogramma con i dati di Emanuele', 'histStroiliData.png')
make_hist(['Michele'], 'Istogramma con i dati di Michele', 'histMichData.png')

Fit status: 0
[56.625   57.08333 56.51667 57.61667 57.11667 56.58333 57.01667 56.7
 56.45    56.89167 57.29167 56.575   56.45    56.53333 56.7     56.45
 56.89167 56.45    56.50833 57.2     56.7     57.2     57.1     56.74167
 56.78333 57.075   56.37    56.53333 57.55    56.61667]
Fit status: 0
Fit status: 0
Fit status: 0
Fit status: 0
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      11.8569
NDf                       =            5
Edm                       =  8.43337e-07
NCalls                    =           74
const                     =      23.2242   +/-   3.70802     
mu                        =      56.7555   +/-   0.0564431   
sigma                     =     0.441517   +/-   0.0779489    	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      2.39484
NDf                       =            1
Edm                       =  4.03727e-07
NCalls                    =        

Info in <TCanvas::Print>: png file hist_totalData.png has been created
Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).
Info in <TCanvas::Print>: png file histManuData.png has been created
Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).
Info in <TCanvas::Print>: png file histAleData.png has been created
Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).
Info in <TCanvas::Print>: png file histStroiliData.png has been created
Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).
Info in <TCanvas::Print>: png file histMichData.png has been created
